<!--
Copyright (c) 2026 Milin Patel
Hochschule Kempten - University of Applied Sciences
Campustage 2026 - Automatisiertes Fahren! Sicher durch Zufall?
MIT License - see LICENSE.
-->

# Der KI-Blick
### Wie ein automatisiertes Fahrzeug Personen erkennt und wo diese Erkennung an Grenzen stößt

Workshop *Automatisiertes Fahren! Sicher durch Zufall?* · Campustage 2026<br>
Hochschule Kempten, Institut für Fahrerassistenz und vernetzte Mobilität (IFM) · Milin Patel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/00_workshop_der_ki_blick.ipynb)

---

### Worum es geht

Automatisierte Fahrzeuge erkennen Personen mit Kameras und einem KI-Modell zur **Objekterkennung**.
Das Modell liefert dabei keine Gewissheit, sondern für jede mögliche Person eine Zahl.
Ab welchem Wert eine Erkennung zählt, legt ein Mensch fest.

In diesem Notebook arbeitest du mit einem echten Modell (YOLOv8n). Du legst diesen Wert selbst fest,
prüfst ihn bei schlechter Sicht und untersuchst einen gezielten Angriff.
Am Ende steht die Frage aus dem Titel: Ist ein solches Fahrzeug *sicher durch Zufall*?

| Teil | Inhalt |
|---|---|
| 1 · Bild und Referenz | Bild wählen, Personen selbst zählen |
| 2 · Objekterkennung | Was das Modell erkennt und was es verwirft |
| 3 · Schwellenwert | Den eigenen Arbeitspunkt festlegen |
| 4 · Schlechte Sicht | Nebel, Regen und Dunkelheit |
| 5 · Zufall oder Absicht | Verdeckung und gezielter Angriff |
| Abschluss | Ergebnis, Fachbegriffe, weiterführendes Material |

Dauer: etwa 20 Minuten. Vorkenntnisse sind nicht nötig.

### Bedienung

1. Im Menü **Laufzeit → Alle ausführen** wählen (englisches Menü: *Runtime → Run all*). Dafür ist ein Google-Konto nötig.
2. Falls Colab einen Hinweis zur Herkunft des Notebooks anzeigt: bestätigen.
3. Warten, bis unter *Vorbereitung* die Meldung **Bereit** erscheint (etwa eine Minute).

Danach wird alles über Schaltflächen und Regler bedient. Der Programmcode ist eingeklappt und muss nicht gelesen werden.

In [ ]:
# @title Vorbereitung: Modell laden
%pip install -q ultralytics==8.4.216
import io, time
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import ipywidgets as W
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils import ASSETS

try:
    from google.colab import output as _colab
    _colab.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---------- Farben, Texte ----------
ORANGE, BLAU, GRAU, LILA = (225, 120, 0), (40, 110, 200), (105, 105, 115), (120, 60, 170)
DE = {"person": "Person", "car": "Auto", "bus": "Bus", "truck": "LKW", "bicycle": "Fahrrad",
      "motorcycle": "Motorrad", "stop sign": "Stoppschild", "traffic light": "Ampel",
      "train": "Zug", "handbag": "Handtasche", "backpack": "Rucksack", "umbrella": "Schirm",
      "skateboard": "Skateboard", "dog": "Hund", "cat": "Katze", "bird": "Vogel",
      "tie": "Krawatte", "cell phone": "Handy", "bench": "Bank", "chair": "Stuhl",
      "fire hydrant": "Hydrant", "parking meter": "Parkuhr", "sports ball": "Ball",
      "suitcase": "Koffer", "bottle": "Flasche", "cup": "Becher", "laptop": "Laptop",
      "potted plant": "Pflanze", "tennis racket": "Tennisschläger", "baseball bat": "Schläger",
      "horse": "Pferd", "kite": "Drachen", "clock": "Uhr", "book": "Buch", "tv": "Bildschirm"}
def de(name): return DE.get(name, name)
def zahl(x):
    """Konfidenz oder Schwellenwert als deutsche Dezimalzahl, abgerundet (0,2497 wird 0,24, nicht 0,25)."""
    return f"{int(100 * x + 1e-6) / 100:.2f}".replace(".", ",")
def sek(x): return f"{x:.2f}".replace(".", ",") + " s"

def karte(text, art="info"):
    farben = {"info": ("#eef4fb", "#1d4e89"), "gut": ("#e9f6ee", "#1e6b3a"),
              "achtung": ("#fdf0e4", "#8a4b0f"), "frage": ("#f3eefa", "#5b2d8a")}
    bg, fg = farben[art]
    return W.HTML(f'<div style="background:{bg};color:{fg};border-left:6px solid {fg};'
                  f'padding:10px 14px;margin:6px 0;border-radius:6px;font-size:15px;'
                  f'line-height:1.5;font-family:sans-serif">{text}</div>')

def kacheln(werte):
    """werte: Liste von (Titel, Zahl, Farbe)."""
    html = "".join(
        f'<div style="flex:1;min-width:110px;background:{f};color:white;border-radius:8px;'
        f'padding:8px 10px;text-align:center;font-family:sans-serif">'
        f'<div style="font-size:13px">{t}</div><div style="font-size:30px;font-weight:bold">{z}</div></div>'
        for t, z, f in werte)
    return W.HTML(f'<div style="display:flex;gap:8px;flex-wrap:wrap;margin:6px 0">{html}</div>')

def knopf(text, stil="primary", breite="240px"):
    return W.Button(description=text, button_style=stil, layout=W.Layout(width=breite, height="38px"))

def sicher(ausgabe):
    """Fehler in Knopf-Funktionen freundlich anzeigen statt still zu verschlucken."""
    def deko(fn):
        def innen(*a, **k):
            try:
                return fn(*a, **k)
            except Exception as e:
                with ausgabe:
                    display(karte(f"Es ist ein Fehler aufgetreten ({type(e).__name__}: {e}). "
                                  "Bitte die Zelle erneut ausführen (▶ links).", "achtung"))
        return innen
    return deko

# ---------- Bilder ----------
def arbeitsbild(img):
    """Auf 640 Pixel (lange Seite) bringen, Seiten durch 32 teilbar."""
    img = img.convert("RGB")
    f = 640 / max(img.size)
    w = max(64, int(round(img.width * f / 32)) * 32)
    h = max(64, int(round(img.height * f / 32)) * 32)
    return img.resize((w, h), Image.LANCZOS)

def jpg(img):
    b = io.BytesIO(); img.save(b, "JPEG", quality=88); return b.getvalue()

def zeige(img, breite="520px"):
    return W.Image(value=jpg(img), format="jpeg", layout=W.Layout(width="100%", max_width=breite))

def nebeneinander(*teile):
    return W.HBox([W.VBox([W.HTML(f'<b style="font-family:sans-serif;font-size:15px">{t}</b>'),
                           zeige(b, "420px")], layout=W.Layout(width=f"{96 // len(teile)}%"))
                   for t, b in teile])

def _schrift(groesse):
    for n in ("DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"):
        try: return ImageFont.truetype(n, groesse)
        except Exception: pass
    try: return ImageFont.load_default(size=groesse)
    except TypeError: return ImageFont.load_default()

def _gestrichelt(d, box, farbe, lw, l=9):
    x1, y1, x2, y2 = box
    for (ax, ay), (bx, by) in (((x1, y1), (x2, y1)), ((x2, y1), (x2, y2)), ((x2, y2), (x1, y2)), ((x1, y2), (x1, y1))):
        n = max(2, int(max(abs(bx - ax), abs(by - ay)) / l))
        for i in range(0, n, 2):
            a, b = i / n, min(1, (i + 1) / n)
            d.line([(ax + (bx - ax) * a, ay + (by - ay) * a), (ax + (bx - ax) * b, ay + (by - ay) * b)], fill=farbe, width=lw)

def _etikett(d, x, y, text, farbe, f):
    l, t, r, b = d.textbbox((0, 0), text, font=f)
    th = b - t
    x = max(0, min(x, d.im.size[0] - (r - l) - 8))
    y = y - th - 8 if y - th - 8 >= 0 else y + 2
    d.rectangle([x, y, x + (r - l) + 8, y + th + 6], fill=farbe)
    d.text((x + 4, y + 3 - t), text, fill=(255, 255, 255), font=f)

def zeichne(img, boxen, grenze, vermutungen=False, nummern=None, nur_menschen=False):
    """Boxen über der Grenze: durchgezogen. Darunter (wenn gewünscht): grau gestrichelt."""
    out = img.copy(); d = ImageDraw.Draw(out)
    f = _schrift(max(13, img.width // 34)); lw = max(2, img.width // 170)
    for b in sorted(boxen, key=lambda b: b["conf"]):
        if nur_menschen and b["name"] != "person":
            continue
        drin = b["conf"] >= grenze
        if not drin and not vermutungen:
            continue
        box = [int(v) for v in b["xyxy"]]
        if drin:
            farbe = ORANGE if b["name"] == "person" else BLAU
            d.rectangle(box, outline=farbe, width=lw)
        else:
            farbe = GRAU
            _gestrichelt(d, box, farbe, max(1, lw - 1))
        _etikett(d, box[0], box[1], f"{de(b['name'])} {zahl(b['conf'])}", farbe, f)
    if nummern:
        fg = _schrift(max(22, img.width // 14))
        for i, b in enumerate(nummern, 1):
            x1, y1, x2, y2 = b["xyxy"]
            cx, cy = (x1 + x2) / 2, y1 + (y2 - y1) * 0.15
            r = max(16, img.width // 26)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=LILA, outline=(255, 255, 255), width=3)
            d.text((cx, cy), str(i), fill=(255, 255, 255), font=fg, anchor="mm")
    return out

# ---------- Modell ----------
MODELL = YOLO("yolov8n.pt")
NAMEN = MODELL.names
STANDARD = 0.30
_cache = {}

def erkenne_alles(img):
    """Einmal mit sehr niedrigem Schwellenwert rechnen; danach wird nur noch gefiltert (darum reagiert der Regler sofort)."""
    key = hash(img.tobytes())
    if key not in _cache:
        if len(_cache) > 60: _cache.clear()
        t = time.perf_counter()
        r = MODELL.predict(img, conf=0.05, imgsz=640, verbose=False)[0]
        dauer = time.perf_counter() - t
        boxen = [dict(name=NAMEN[int(b.cls[0])], conf=float(b.conf[0]), xyxy=[float(v) for v in b.xyxy[0]])
                 for b in r.boxes]
        _cache[key] = (boxen, dauer)
    return _cache[key]

def personen(boxen, grenze):
    return [b for b in boxen if b["name"] == "person" and b["conf"] >= grenze]

def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    iw, ih = max(0, min(ax2, bx2) - max(ax1, bx1)), max(0, min(ay2, by2) - max(ay1, by1))
    i = iw * ih
    return i / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - i + 1e-9)

def bester_treffer(boxen, ziel):
    """Höchste Konfidenz einer Person an der Stelle des Ziels (0, wenn dort keine mehr erkannt wird)."""
    werte = [b["conf"] for b in boxen if b["name"] == "person" and iou(b["xyxy"], ziel["xyxy"]) > 0.3]
    return max(werte, default=0.0)

S = dict(bild=None, quelle=None, gezaehlt=None, zeit_mensch=None, zeit_modell=None, modell_personen=None,
         grenze=None, zielkonflikt=None, wetter_tipp=None, wetter_ergebnis=None, nebel_entscheidung=None,
         angriff=None, antwort_vorher=None, antwort_nachher=None)

def grenze(): return S["grenze"] if S["grenze"] is not None else STANDARD

def neues_bild(img, quelle):
    S.update(bild=arbeitsbild(img), quelle=quelle, gezaehlt=None, zeit_mensch=None, zeit_modell=None,
             modell_personen=None, wetter_ergebnis=None, angriff=None)

# ---------- Wetter (simuliert) ----------
def wetter(img, art, staerke):
    """staerke von 0 bis 1."""
    if staerke <= 0 or art == "Klar":
        return img
    rng = np.random.default_rng(7)
    if art == "Nebel":
        grau = Image.new("RGB", img.size, (208, 208, 212))
        return Image.blend(img, grau, 0.88 * staerke).filter(ImageFilter.GaussianBlur(0.5 + 2.5 * staerke))
    if art == "Regen":
        o = img.copy(); d = ImageDraw.Draw(o); w, h = img.size
        lang = max(8, h // 30)
        for _ in range(int(w * h / 150 * staerke)):
            x, y = int(rng.integers(0, w)), int(rng.integers(0, h))
            d.line([(x, y), (x - lang // 5, y + lang)], fill=(195, 200, 215), width=2)
        o = o.filter(ImageFilter.GaussianBlur(0.6 + 1.2 * staerke))
        return Image.eval(o, lambda p: int(p * (1 - 0.35 * staerke)))
    if art == "Dunkelheit":
        a = np.asarray(img).astype(np.float32) * (1 - 0.93 * staerke)
        a += rng.normal(0, 18 * staerke, a.shape)          # Kamera-Rauschen wie bei echten Nachtbildern
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    return img

# ---------- Aufkleber ----------
def aufkleber_ort(img, ziel, anteil):
    x1, y1, x2, y2 = ziel["xyxy"]
    g = int(max(12, anteil * (y2 - y1)))
    g = min(g, img.width, img.height)
    cx = int((x1 + x2) / 2); cy = int(y1 + 0.38 * (y2 - y1))
    px = min(max(0, cx - g // 2), img.width - g); py = min(max(0, cy - g // 2), img.height - g)
    return px, py, g

def zufalls_aufkleber(img, ziel, anteil, rng):
    px, py, g = aufkleber_ort(img, ziel, anteil)
    muster = Image.fromarray(rng.integers(0, 256, (g, g, 3), dtype=np.uint8))
    o = img.copy(); o.paste(muster, (px, py)); return o, muster

def berechne_aufkleber(img, ziel, anteil, schritte=40, fortschritt=None):
    """Gezielter Angriff (white-box): In jedem Schritt wird berechnet, in welche Richtung jedes Pixel
    des Musters geändert werden muss, damit die Konfidenz für 'Person' an dieser Stelle sinkt."""
    px, py, g = aufkleber_ort(img, ziel, anteil)
    net = MODELL.model.eval()
    dev = next(net.parameters()).device
    dtype = next(net.parameters()).dtype
    for p in net.parameters(): p.requires_grad_(False)
    x0 = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev)
    x1, y1, x2, y2 = ziel["xyxy"]
    patch = torch.rand(1, 3, g, g, device=dev, requires_grad=True)
    for i in range(schritte):
        x = x0.clone(); x[:, :, py:py + g, px:px + g] = patch
        out = net(x.to(dtype)); out = out[0] if isinstance(out, (list, tuple)) else out
        bx, sc = out[0, :4].float(), out[0, 4].float()          # Zeile 4 = Klasse 'person'
        drin = (bx[0] > x1) & (bx[0] < x2) & (bx[1] > y1) & (bx[1] < y2)
        wert = sc[drin].max() if bool(drin.any()) else sc.max()
        wert.backward()
        with torch.no_grad():
            patch -= 0.03 * patch.grad.sign(); patch.clamp_(0, 1); patch.grad.zero_()
        if fortschritt is not None: fortschritt.value = i + 1
    muster = Image.fromarray((patch.detach()[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8))
    o = img.copy(); o.paste(muster, (px, py)); return o, muster

# ---------- Start ----------
neues_bild(Image.open(ASSETS / "bus.jpg"), "Beispielbild (Straßenszene)")
erkenne_alles(S["bild"]); _cache.clear(); erkenne_alles(S["bild"])   # Aufwärmen, damit die Zeitmessung fair ist
display(karte("<b>Bereit.</b> Das Modell ist geladen. Weiter mit <b>Teil 1</b>.", "gut"))

---
## Teil 1 · Bild und Referenz

Das Modell arbeitet mit einem Kamerabild. Voreingestellt ist eine Straßenszene.
Du kannst stattdessen ein eigenes Foto hochladen, zum Beispiel ein Bild aus der Kamera des Roboters.
Geeignet sind Fotos, auf denen Personen zu sehen sind.

In [ ]:
# @title Teil 1a · Bild auswählen
t1_out = W.Output()
t1_upload = W.FileUpload(accept="image/*", multiple=False, description="Foto auswählen")
t1_nimm = knopf("Foto verwenden", breite="170px")
t1_std = knopf("Beispielbild verwenden", stil="", breite="200px")

def t1_zeigen(text, art="info"):
    with t1_out:
        t1_out.clear_output(wait=True)
        display(karte(text, art)); display(zeige(S["bild"]))

def _hochgeladen(u):
    v = u.value
    if not v: return None
    if isinstance(v, dict): return list(v.values())[0]["content"]
    return bytes(v[0]["content"])

@sicher(t1_out)
def t1_nimm_klick(_):
    daten = _hochgeladen(t1_upload)
    if daten is None:
        t1_zeigen("Es wurde noch kein Foto ausgewählt. Bitte zuerst <b>Foto auswählen</b> anklicken.", "achtung"); return
    neues_bild(Image.open(io.BytesIO(daten)), "eigenes Foto")
    t1_zeigen("<b>Eigenes Foto geladen.</b> Alle folgenden Teile verwenden dieses Bild.", "gut")

@sicher(t1_out)
def t1_std_klick(_):
    neues_bild(Image.open(ASSETS / "bus.jpg"), "Beispielbild (Straßenszene)")
    t1_zeigen("Das Beispielbild ist ausgewählt.")

t1_nimm.on_click(t1_nimm_klick); t1_std.on_click(t1_std_klick)
display(W.HBox([t1_upload, t1_nimm, t1_std]), t1_out)
t1_zeigen(f"Aktuelles Bild: <b>{S['quelle']}</b>")

Zähle jetzt selbst: **Wie viele Personen sind auf dem Bild?**
Deine Zählung ist die **Referenz** (*Ground Truth*). Mit ihr wird das Ergebnis des Modells verglichen.
Die Zeit wird mitgemessen.

In [ ]:
# @title Teil 1b · Personen zählen
t1b_out = W.Output(); t1b_bild = W.Output()
t1b_start = knopf("Zählung starten", breite="170px")
t1b_zahl = W.BoundedIntText(value=0, min=0, max=99, description="Personen:", layout=W.Layout(width="170px"))
t1b_fertig = knopf("Zählung abschließen", stil="success", breite="190px")
_t1b = {"t0": None}

@sicher(t1b_out)
def t1b_start_klick(_):
    _t1b["t0"] = time.perf_counter()
    with t1b_bild:
        t1b_bild.clear_output(wait=True); display(zeige(S["bild"]))
    with t1b_out:
        t1b_out.clear_output(); display(karte("Die Zeit läuft.", "frage"))

@sicher(t1b_out)
def t1b_fertig_klick(_):
    S["gezaehlt"] = int(t1b_zahl.value)
    S["zeit_mensch"] = time.perf_counter() - _t1b["t0"] if _t1b["t0"] else None
    _t1b["t0"] = None
    zeit = f" in {sek(S['zeit_mensch'])}" if S["zeit_mensch"] else ""
    with t1b_out:
        t1b_out.clear_output()
        display(karte(f"Referenz: <b>{S['gezaehlt']} Personen</b>{zeit}.", "gut"))

t1b_start.on_click(t1b_start_klick); t1b_fertig.on_click(t1b_fertig_klick)
display(t1b_start, t1b_bild, W.HBox([t1b_zahl, t1b_fertig]), t1b_out)

---
## Teil 2 · Objekterkennung

Das Modell untersucht das ganze Bild und berechnet für viele Bildausschnitte, wie stark sie einer Person,
einem Bus oder einem von insgesamt 80 Objekttypen ähneln. Das Ergebnis sind **Erkennungskästen** (*Bounding Boxes*),
jeweils mit einer **Konfidenz** (*Confidence*) zwischen 0 und 1.

Die Konfidenz ist ein vom Modell berechneter Wert, **keine Wahrscheinlichkeit**.
Eine Konfidenz von 0,86 bedeutet nicht, dass die Erkennung in 86 von 100 Fällen stimmt.

Angezeigt wird eine Erkennung nur, wenn ihre Konfidenz mindestens **0,30** beträgt.
Diesen Wert nennt man **Schwellenwert** (*Threshold*).

In [ ]:
# @title Teil 2 · Erkennung starten
t2_out = W.Output()
t2_los = knopf("Erkennung starten", breite="180px")
t2_kand = W.ToggleButton(value=False, description="Verworfene Kandidaten anzeigen",
                         layout=W.Layout(width="270px", height="38px"), disabled=True)

def t2_zeichnen():
    boxen, dauer = erkenne_alles(S["bild"])
    n = len(personen(boxen, STANDARD))
    S["zeit_modell"], S["modell_personen"] = dauer, n
    with t2_out:
        t2_out.clear_output(wait=True)
        if S["gezaehlt"] is None:
            text = f"<b>Modell:</b> {n} Personen in {sek(dauer)}. (Für den Vergleich zuerst in Teil 1 zählen.)"
        else:
            zeit = f" in {sek(S['zeit_mensch'])}" if S["zeit_mensch"] else ""
            text = (f"<b>Referenz:</b> {S['gezaehlt']} Personen{zeit} &nbsp;·&nbsp; "
                    f"<b>Modell:</b> {n} Personen in {sek(dauer)}<br>")
            text += ("Die Anzahl stimmt überein. Prüfe trotzdem, ob es dieselben Personen sind."
                     if n == S["gezaehlt"] else "Die Anzahl weicht ab. Welche Person wird unterschiedlich bewertet?")
        display(karte(text, "frage"))
        display(zeige(zeichne(S["bild"], boxen, STANDARD, vermutungen=t2_kand.value)))
        if t2_kand.value:
            unter = sorted([b for b in boxen if b["conf"] < STANDARD], key=lambda b: -b["conf"])
            liste = ", ".join(f"{de(b['name'])} {zahl(b['conf'])}" for b in unter[:8]) or "keine"
            text = (f"<b>Grau gestrichelt:</b> Kandidaten unter dem Schwellenwert von {zahl(STANDARD)}: {liste}.<br>"
                    "Das Modell hat sie berechnet, aber verworfen. Viele davon sind falsch.")
            pers = [b for b in unter if b["name"] == "person" and b["conf"] >= 0.15]
            if pers:
                text += (f" Darunter ist aber auch eine Person mit der Konfidenz {zahl(pers[0]['conf'])}. "
                         "Ob sie erkannt wird, hängt allein vom Schwellenwert ab.")
            display(karte(text, "achtung"))

@sicher(t2_out)
def t2_klick(_):
    t2_kand.disabled = False; t2_zeichnen()

t2_los.on_click(t2_klick)
t2_kand.observe(lambda c: sicher(t2_out)(t2_zeichnen)(), names="value")
display(W.HBox([t2_los, t2_kand]), t2_out)

Eine Frage, bevor es weitergeht. Die Antwort wird gespeichert und im Abschluss wieder aufgegriffen.

In [ ]:
# @title Teil 2 · Frage
FRAGE = "Wer hat festgelegt, dass der Schwellenwert 0,30 beträgt?"
OPTIONEN = ["Das KI-Modell selbst", "Der Zufall", "Ein Mensch"]
t2f_out = W.Output()
t2f_knoepfe = [knopf(o, stil="", breite="200px") for o in OPTIONEN]
def t2f_klick(b):
    S["antwort_vorher"] = b.description
    for k in t2f_knoepfe: k.disabled = True
    with t2f_out:
        t2f_out.clear_output(); display(karte(f"Gespeichert: <i>{b.description}</i>. Teil 3 geht dieser Frage nach.", "info"))
for k in t2f_knoepfe: k.on_click(t2f_klick)
display(karte(f"<b>{FRAGE}</b>", "frage"), W.HBox(t2f_knoepfe), t2f_out)

---
## Teil 3 · Schwellenwert festlegen

Ein hoher Schwellenwert unterdrückt falsche Meldungen, lässt aber Personen unerkannt.
Ein niedriger Schwellenwert erkennt mehr Personen, erzeugt aber mehr falsche Meldungen.
Diesen **Zielkonflikt** (*Trade-off*) gibt es in jedem Erkennungssystem.

- **Übersehene Person** (*False Negative*): Eine Person ist im Bild, wird aber nicht erkannt.
- **Fehlalarm** (*False Positive*): Das Modell meldet eine Person, wo keine ist.

Stelle den Schwellenwert so ein, wie du ihn für ein Fahrzeug wählen würdest. Probiere vorher auch die Extremwerte aus.
Lege den Wert dann als deinen **Arbeitspunkt** (*Operating Point*) fest.

In [ ]:
# @title Teil 3 · Schwellenwert
t3_out = W.Output(); t3_info = W.Output(); t3_konflikt = W.Output()
WERTE = [round(0.05 * i, 2) for i in range(1, 20)]
t3_regler = W.SelectionSlider(options=[(zahl(v), v) for v in WERTE], value=STANDARD, description="Schwellenwert",
                              continuous_update=False, style={"description_width": "110px"}, layout=W.Layout(width="440px"))
t3_fest = knopf("Arbeitspunkt festlegen", stil="success", breite="220px")

def t3_zeichnen(*_):
    g = t3_regler.value
    boxen, _d = erkenne_alles(S["bild"])
    n = len(personen(boxen, g))
    andere = sorted({de(b["name"]) for b in boxen if b["name"] != "person" and b["conf"] >= g})
    with t3_out:
        t3_out.clear_output(wait=True)
        if S["gezaehlt"] is None:
            display(kacheln([("Erkannte Personen", n, "#555")]))
            display(karte("Für den Vergleich mit der Referenz zuerst in Teil 1 zählen.", "info"))
        else:
            fn, fp = max(0, S["gezaehlt"] - n), max(0, n - S["gezaehlt"])
            display(kacheln([("Erkannte Personen", n, "#555"),
                             ("Übersehen (False Negative)", fn, "#b03a2e" if fn else "#2e7d4f"),
                             ("Fehlalarm (False Positive)", fp, "#b9770e" if fp else "#2e7d4f")]))
            display(W.HTML('<div style="font-family:sans-serif;font-size:13px;color:#666">Verglichen wird nur die Anzahl. '
                           'Prüfe an den Kästen, ob wirklich die richtigen Personen erkannt werden.</div>'))
        if andere:
            display(W.HTML(f'<div style="font-family:sans-serif;font-size:14px">Weitere Erkennungen: {", ".join(andere)}</div>'))
        display(zeige(zeichne(S["bild"], boxen, g)))

@sicher(t3_info)
def t3_fest_klick(_):
    g = t3_regler.value; S["grenze"] = g
    boxen, _d = erkenne_alles(S["bild"]); n = len(personen(boxen, g))
    von = f" von {S['gezaehlt']}" if S["gezaehlt"] is not None else ""
    with t3_info:
        t3_info.clear_output()
        display(karte(f"<b>Arbeitspunkt festgelegt: {zahl(g)}.</b> Bei diesem Bild erkennt das Modell damit {n}{von} Personen.<br><br>"
                      "Diesen Wert hat weder das Modell noch der Zufall bestimmt, sondern du. "
                      "In der Fahrzeugentwicklung treffen Ingenieurinnen und Ingenieure diese Entscheidung. "
                      "Sie begründen sie mit Messungen auf großen Datensätzen, nicht mit einem einzelnen Bild.", "gut"))
    with t3_konflikt:
        t3_konflikt.clear_output()
        a = knopf("Eine übersehene Person", stil="", breite="230px")
        b = knopf("Ein Fehlalarm", stil="", breite="230px")
        antwort = W.Output()
        ende = ("<br><br>Einen Schwellenwert, der beides vermeidet, gibt es nicht. Viele automatisierte Fahrzeuge "
                "kombinieren deshalb mehrere Sensoren (Kamera, Radar, LiDAR) und werten Erkennungen über mehrere "
                "aufeinanderfolgende Bilder aus.")
        def wahl(text, schluessel):
            S["zielkonflikt"] = schluessel
            with antwort:
                antwort.clear_output(); display(karte(text + ende, "info"))
        a.on_click(lambda _: wahl("Eine übersehene Person kann schwer verletzt werden. Häufige Fehlalarme sind aber ebenfalls ein Risiko: "
                                  "Sie können zu unnötigen Bremsungen führen, die nachfolgende Fahrzeuge gefährden.", "übersehen"))
        b.on_click(lambda _: wahl("Fehlalarme können zu unnötigen Bremsungen führen und nachfolgende Fahrzeuge gefährden. "
                                  "Eine übersehene Person hat jedoch meist die schwereren Folgen.", "fehlalarm"))
        display(karte("<b>Zielkonflikt:</b> Was wiegt für ein Fahrzeug schwerer?", "frage"), W.HBox([a, b]), antwort)

t3_regler.observe(lambda c: sicher(t3_out)(t3_zeichnen)(), names="value")
t3_fest.on_click(t3_fest_klick)
display(W.HBox([t3_regler, t3_fest]), t3_info, t3_out, t3_konflikt)
t3_zeichnen()

---
## Teil 4 · Schlechte Sicht

Nebel, Regen und Dunkelheit sind typische **auslösende Bedingungen** (*Triggering Conditions*):
Umstände, unter denen ein System Fehler macht, obwohl kein Bauteil defekt ist.
Mit solchen Fällen befasst sich die Norm ISO 21448 (*SOTIF, Safety of the Intended Functionality*).

Die Bedingungen werden hier am Computer simuliert. Es gilt dein Arbeitspunkt aus Teil 3.

**Zuerst deine Einschätzung:** Welche Bedingung beeinträchtigt die Erkennung am stärksten?

In [ ]:
# @title Teil 4 · Schlechte Sicht
t4_out = W.Output(); t4_ergebnis = W.Output(); t4_entscheidung = W.Output()
ARTEN = ["Nebel", "Regen", "Dunkelheit"]
t4_tipp = W.ToggleButtons(options=ARTEN, value="Nebel")
t4_tipp_ok = knopf("Einschätzung speichern", breite="210px")
t4_art = W.ToggleButtons(options=ARTEN, value="Nebel")
t4_staerke = W.IntSlider(value=50, min=0, max=100, step=10, description="Stärke (%)", continuous_update=False,
                         style={"description_width": "90px"}, layout=W.Layout(width="380px"))
t4_auswerten = knopf("Auswerten", stil="success", breite="160px")
t4_test = W.VBox([karte("Wähle eine Bedingung und ihre Stärke. Wenn du alle drei geprüft hast: <b>Auswerten</b>.", "info"),
                  W.HBox([t4_art, t4_staerke]), t4_out, t4_auswerten, t4_ergebnis, t4_entscheidung],
                 layout=W.Layout(display="none"))

def t4_zeichnen(*_):
    g = grenze(); klar = S["bild"]; schlecht = wetter(klar, t4_art.value, t4_staerke.value / 100)
    bk, _a = erkenne_alles(klar); bs, _b = erkenne_alles(schlecht)
    with t4_out:
        t4_out.clear_output(wait=True)
        display(W.HTML(f'<div style="font-family:sans-serif;font-size:14px">Schwellenwert: <b>{zahl(g)}</b>'
                       f'{" (Arbeitspunkt aus Teil 3)" if S["grenze"] is not None else " (Voreinstellung)"}</div>'))
        display(kacheln([("Klare Sicht: Personen", len(personen(bk, g)), "#2e7d4f"),
                         (f"{t4_art.value}: Personen", len(personen(bs, g)), "#555")]))
        display(nebeneinander(("Klare Sicht", zeichne(klar, bk, g)),
                              (f"{t4_art.value}, Stärke {t4_staerke.value} %", zeichne(schlecht, bs, g))))

def _verlust(art, staerke, ziele, g):
    bs, _b = erkenne_alles(wetter(S["bild"], art, staerke))
    return sum(1 for z in ziele if bester_treffer(bs, z) < g), sum(z["conf"] - bester_treffer(bs, z) for z in ziele)

@sicher(t4_out)
def t4_tipp_klick(_):
    S["wetter_tipp"] = t4_tipp.value; t4_tipp.disabled = True; t4_tipp_ok.disabled = True
    t4_test.layout.display = "flex"; t4_zeichnen()

@sicher(t4_ergebnis)
def t4_auswerten_klick(_):
    g = grenze(); bk, _a = erkenne_alles(S["bild"]); ziele = personen(bk, g)
    if not ziele:
        with t4_ergebnis:
            t4_ergebnis.clear_output(); display(karte("Bei klarer Sicht wird keine Person erkannt. Bitte ein anderes Bild wählen.", "achtung"))
        return
    res = {a: _verlust(a, 0.7, ziele, g) for a in ARTEN}
    schlimmst = max(res, key=lambda a: res[a])
    S["wetter_ergebnis"] = {a: v[0] for a, v in res.items()} | {"schlimmst": schlimmst}
    with t4_ergebnis:
        t4_ergebnis.clear_output(wait=True)
        display(kacheln([(f"{a}: nicht mehr erkannt", v[0], "#b03a2e" if a == schlimmst else "#777") for a, v in res.items()]))
        if res[schlimmst][0] > 0:
            text = (f"Bei 70 % Stärke beeinträchtigt <b>{schlimmst}</b> die Erkennung am stärksten: "
                    f"Von {len(ziele)} Personen werden {res[schlimmst][0]} nicht mehr erkannt. ")
        else:
            text = (f"Bei 70 % Stärke wird keine Person ganz verloren. Am stärksten sinkt die Konfidenz bei <b>{schlimmst}</b>. ")
        text += ("Deine Einschätzung war zutreffend." if S["wetter_tipp"] == schlimmst
                 else f"Deine Einschätzung war: {S['wetter_tipp']}.")
        if res["Dunkelheit"][0] == 0 and schlimmst != "Dunkelheit" and max(v[0] for v in res.values()) > 0:
            text += ("<br><br>In dieser Simulation beeinträchtigt Dunkelheit das Modell kaum, während sie für Menschen eine starke Einschränkung ist. "
                     "Modell und Mensch haben unterschiedliche Stärken und Schwächen.")
        text += ("<br><br><i>Gut zu wissen:</i> Die Simulation ist vereinfacht. Echter Nebel, echte Nacht und Blendung durch Scheinwerfer "
                 "sind anspruchsvoller. Fahrzeuge werden deshalb in sehr vielen Situationen getestet, auf Testgeländen und in Simulationen.")
        display(karte(text, "gut"))
    # Zweite Entscheidung: Ab welcher Stärke verliert der Arbeitspunkt eine Person?
    stufe = next((s for s in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0) if _verlust(schlimmst, s, ziele, g)[0] > 0), None)
    with t4_entscheidung:
        t4_entscheidung.clear_output()
        if stufe is None:
            display(karte(f"Mit deinem Arbeitspunkt {zahl(g)} geht in dieser Simulation selbst bei 100 % {schlimmst} keine Person verloren. "
                          "Prüfe einen höheren Schwellenwert in Teil 3.", "info")); return
        verloren = _verlust(schlimmst, stufe, ziele, g)[0]
        bs, _b = erkenne_alles(wetter(S["bild"], schlimmst, stufe))
        neu = next((w for w in sorted(WERTE, reverse=True) if w < g and all(bester_treffer(bs, z) >= w for z in ziele)), None)
        a = knopf("Schwellenwert senken", stil="", breite="210px")
        b = knopf("Einsatzbereich begrenzen", stil="", breite="230px")
        antwort = W.Output()
        def senken(_):
            S["nebel_entscheidung"] = "senken"
            with antwort:
                antwort.clear_output()
                if neu is None:
                    display(karte("Selbst mit dem niedrigsten Schwellenwert werden nicht alle Personen wieder erkannt. "
                                  "Hier hilft nur, den Einsatzbereich zu begrenzen.", "info")); return
                zusatz = max(0, len(personen(bk, neu)) - (S["gezaehlt"] if S["gezaehlt"] is not None else len(ziele)))
                display(karte(f"Mit dem Schwellenwert {zahl(neu)} würden bei dieser Bedingung wieder alle {len(ziele)} Personen erkannt. "
                              f"Bei klarer Sicht ergibt derselbe Wert {zusatz} Fehlalarm{'e' if zusatz != 1 else ''} gegenüber deiner Referenz. "
                              "Der Zielkonflikt verschiebt sich, er verschwindet nicht.", "info"))
        def begrenzen(_):
            S["nebel_entscheidung"] = "begrenzen"
            with antwort:
                antwort.clear_output()
                display(karte("Das ist in der Praxis der übliche Weg. Der Hersteller legt den <b>Einsatzbereich</b> fest "
                              "(<i>Operational Design Domain, ODD</i>): die Bedingungen, für die das System ausgelegt ist, "
                              f"zum Beispiel „nicht bei starkem {schlimmst}“. Erkennt das Fahrzeug, dass es diesen Bereich verlässt, "
                              "übergibt es an die Person am Steuer oder führt ein <b>risikominimales Manöver</b> "
                              "(<i>Minimal Risk Maneuver</i>) aus, etwa ein kontrolliertes Anhalten.", "info"))
        a.on_click(sicher(antwort)(senken)); b.on_click(sicher(antwort)(begrenzen))
        display(karte(f"<b>Neue Lage:</b> Bei {schlimmst} mit {round(stufe * 100)} % Stärke erkennt das Modell mit deinem Arbeitspunkt "
                      f"{zahl(g)} {verloren} Person{'en' if verloren != 1 else ''} nicht mehr. Wie reagierst du?", "frage"),
                W.HBox([a, b]), antwort)

t4_tipp_ok.on_click(t4_tipp_klick); t4_auswerten.on_click(t4_auswerten_klick)
for w in (t4_art, t4_staerke): w.observe(lambda c: sicher(t4_out)(t4_zeichnen)(), names="value")
display(W.HBox([t4_tipp, t4_tipp_ok]), t4_test)

---
## Teil 5 · Zufall oder Absicht

Kann ein Muster auf der Kleidung eine Person für das Modell unsichtbar machen? Zwei Versuche mit gleich großen Mustern:

1. **Zufälliges Muster.** Es verdeckt einen Teil der Person (**Verdeckung**, *Occlusion*).
   Du kannst den Versuch beliebig oft wiederholen, jedes Mal mit einem neuen Zufallsmuster.
2. **Berechnetes Muster.** In 50 Schritten wird jeweils berechnet, in welche Richtung jedes Pixel des Musters
   geändert werden muss, damit die Konfidenz für „Person“ sinkt. Dafür muss der innere Aufbau des Modells bekannt sein.
   Man spricht von einem **gezielten Angriff** (*Adversarial Attack*). Die Berechnung dauert einige Sekunden.

In [ ]:
# @title Teil 5 · Zufall oder Absicht
t5_out = W.Output(); t5_ergebnis = W.Output()
t5_ziel = W.Dropdown(description="Person:", layout=W.Layout(width="190px"))
t5_groesse = W.IntSlider(value=30, min=10, max=45, step=5, description="Mustergröße (%)", continuous_update=False,
                         style={"description_width": "120px"}, layout=W.Layout(width="400px"))
t5_zufall = knopf("Zufälliges Muster", stil="", breite="190px")
t5_rechnen = knopf("Berechnetes Muster", stil="", breite="200px")
t5_speichern = knopf("Muster speichern", stil="", breite="170px")
t5_balken = W.IntProgress(value=0, min=0, max=50, description="Berechnung:", style={"description_width": "90px"},
                          layout=W.Layout(width="400px", visibility="hidden"))
_t5 = {"ziele": [], "grenze": None, "versuche": 0, "erfolge": 0, "rng": np.random.default_rng(), "kleinste": None, "muster": None}

def t5_vorbereiten():
    boxen, _d = erkenne_alles(S["bild"])
    ziele = sorted(personen(boxen, grenze()), key=lambda b: b["xyxy"][0])
    _t5.update(ziele=ziele, grenze=grenze(), versuche=0, erfolge=0, kleinste=None, muster=None)
    t5_ziel.options = [(f"Person {i}", i - 1) for i in range(1, len(ziele) + 1)] or [("keine Person erkannt", -1)]
    t5_ziel.value = 0 if ziele else -1
    with t5_out:
        t5_out.clear_output(wait=True)
        if ziele:
            display(karte(f"Wähle eine Person und die Größe des Musters (in Prozent der Personengröße). "
                          f"Es gilt der Schwellenwert {zahl(grenze())}.", "info"))
            display(zeige(zeichne(S["bild"], boxen, grenze(), nummern=ziele, nur_menschen=True)))
        else:
            display(karte("Auf diesem Bild wird keine Person erkannt. Bitte in Teil 1 ein anderes Bild wählen.", "achtung"))

def _bereit():
    if _t5["grenze"] != grenze() or not _t5["ziele"]:
        t5_vorbereiten()
    return t5_ziel.value is not None and t5_ziel.value >= 0

def _pruefen(bild_neu):
    z = _t5["ziele"][t5_ziel.value]
    bn, _d = erkenne_alles(bild_neu)
    nachher = bester_treffer(bn, z)
    return bn, z["conf"], nachher, nachher < grenze()

@sicher(t5_ergebnis)
def t5_zufall_klick(_):
    if not _bereit(): return
    z = _t5["ziele"][t5_ziel.value]
    neu, _m = zufalls_aufkleber(S["bild"], z, t5_groesse.value / 100, _t5["rng"])
    bn, vorher, nachher, weg = _pruefen(neu)
    _t5["versuche"] += 1; _t5["erfolge"] += int(weg)
    with t5_ergebnis:
        t5_ergebnis.clear_output(wait=True)
        display(kacheln([("Versuche", _t5["versuche"], "#555"), ("Person nicht mehr erkannt", _t5["erfolge"], "#b03a2e"),
                         ("Konfidenz vorher → nachher", f"{zahl(vorher)} → {zahl(nachher)}", "#555")]))
        display(zeige(zeichne(neu, bn, grenze(), nur_menschen=True)))
        display(karte("Das zufällige Muster hat in diesem Versuch gewirkt. Das ist selten. Wiederhole den Versuch."
                      if weg else "Die Person wird weiterhin erkannt. Die Verdeckung senkt die Konfidenz nur wenig.", "info"))

@sicher(t5_ergebnis)
def t5_rechnen_klick(_):
    if not _bereit(): return
    z = _t5["ziele"][t5_ziel.value]; a = t5_groesse.value / 100
    t5_balken.layout.visibility = "visible"; t5_balken.value = 0; t5_rechnen.disabled = True
    try:
        neu, muster = berechne_aufkleber(S["bild"], z, a, schritte=t5_balken.max, fortschritt=t5_balken)
    finally:
        t5_rechnen.disabled = False; t5_balken.layout.visibility = "hidden"
    zuf, _m = zufalls_aufkleber(S["bild"], z, a, np.random.default_rng(1))
    bn, vorher, nachher, weg = _pruefen(neu)
    bz, _v, nach_z, _w = _pruefen(zuf)
    _t5["muster"] = muster
    if weg and (_t5["kleinste"] is None or t5_groesse.value < _t5["kleinste"]): _t5["kleinste"] = t5_groesse.value
    S["angriff"] = dict(groesse=t5_groesse.value, wirksam=weg, kleinste=_t5["kleinste"])
    with t5_ergebnis:
        t5_ergebnis.clear_output(wait=True)
        display(nebeneinander((f"Zufällig: Konfidenz {zahl(nach_z)}", zeichne(zuf, bz, grenze(), nur_menschen=True)),
                              (f"Berechnet: Konfidenz {zahl(nachher)}", zeichne(neu, bn, grenze(), nur_menschen=True))))
        if weg:
            display(karte(f"<b>Die Person wird nicht mehr erkannt.</b> Ihre Konfidenz sinkt von {zahl(vorher)} auf {zahl(nachher)} "
                          f"und liegt damit unter dem Schwellenwert {zahl(grenze())}. Für einen Menschen ist sie weiterhin deutlich sichtbar.<br>"
                          f"Kleinste wirksame Mustergröße bisher: <b>{_t5['kleinste']} %</b>. Wie klein darf das Muster sein, damit der Angriff noch wirkt?", "achtung"))
        else:
            display(karte(f"Die Konfidenz sinkt von {zahl(vorher)} auf {zahl(nachher)}, liegt aber noch über dem Schwellenwert. "
                          "Vergrößere das Muster und berechne es erneut.", "info"))
        display(karte("<b>Einordnung:</b> Das zufällige Muster verdeckt nur einen Teil der Person. Das berechnete Muster nutzt "
                      "gezielt Schwächen des Modells aus. Zufällige Störungen sind selten so wirksam wie gezielte.<br><br>"
                      "<i>Gut zu wissen:</i> Dies ist ein digitaler Angriff auf ein bekanntes Modell. Ein gedrucktes Muster muss bei "
                      "wechselndem Licht und aus verschiedenen Blickwinkeln wirken und ist deutlich schwerer umzusetzen. "
                      "Forschungsarbeiten zeigen, dass es grundsätzlich möglich ist, etwa mit Aufklebern auf Verkehrszeichen "
                      "(Eykholt et al., 2018) oder mit einem gedruckten Muster gegen einen Personendetektor (Thys et al., 2019). "
                      "Der Schutz vor solchen Angriffen ist Aufgabe der <b>Security</b>.", "info"))

@sicher(t5_ergebnis)
def t5_speichern_klick(_):
    if _t5["muster"] is None:
        with t5_ergebnis: display(karte("Zuerst ein Muster berechnen.", "achtung")); return
    _t5["muster"].resize((600, 600), Image.NEAREST).save("berechnetes_muster.png")
    if IN_COLAB:
        from google.colab import files; files.download("berechnetes_muster.png")
    else:
        with t5_ergebnis: display(karte("Gespeichert als <code>berechnetes_muster.png</code>.", "gut"))

t5_zufall.on_click(t5_zufall_klick); t5_rechnen.on_click(t5_rechnen_klick); t5_speichern.on_click(t5_speichern_klick)
t5_neu = knopf("Bild neu laden", stil="", breite="150px"); t5_neu.on_click(lambda _: sicher(t5_out)(t5_vorbereiten)())
display(W.HBox([t5_ziel, t5_groesse, t5_neu]), t5_out, W.HBox([t5_zufall, t5_rechnen, t5_speichern]), t5_balken, t5_ergebnis)
t5_vorbereiten()

---
## Abschluss · Sicher durch Zufall?

Zum Schluss noch einmal die Frage aus Teil 2.

In [ ]:
# @title Abschluss
ab_out = W.Output(); ab_zusammenfassung = W.Output()
ab_knoepfe = [knopf(o, stil="", breite="200px") for o in OPTIONEN]
def ab_klick(b):
    S["antwort_nachher"] = b.description
    for k in ab_knoepfe: k.disabled = True
    vorher = S["antwort_vorher"] or "nicht beantwortet"
    with ab_out:
        ab_out.clear_output()
        display(kacheln([("Antwort in Teil 2", vorher, "#777"), ("Antwort jetzt", b.description, "#2e7d4f" if b.description == "Ein Mensch" else "#555")]))
        display(karte("Den Schwellenwert hat ein Mensch festgelegt: Die Voreinstellung 0,30 stammt von den Entwicklern der Software, "
                      "deinen Arbeitspunkt in Teil 3 hast du selbst gewählt. Das Modell berechnet nur die Konfidenz.", "gut"))
        display(karte("<b>Sicher durch Zufall? Nein.</b> Der Zufall bestimmt, welche Situationen auftreten: Wetter, Licht, verdeckte Personen. "
                      "Wie ein Fahrzeug damit umgeht, entscheiden Menschen: mit dem Arbeitspunkt, mit dem festgelegten Einsatzbereich, "
                      "mit Tests und mit dem Schutz gegen Angriffe. Diese Entscheidungen müssen nachvollziehbar begründet werden, "
                      "zum Beispiel in einem <b>Sicherheitsnachweis</b> (<i>Safety Case</i>).", "frage"))
    p = []
    if S["gezaehlt"] is not None: p.append(f"Referenz: {S['gezaehlt']} Personen; Modell bei 0,30: {S['modell_personen'] if S['modell_personen'] is not None else '–'} Personen.")
    if S["grenze"] is not None: p.append(f"Dein Arbeitspunkt: {zahl(S['grenze'])}.")
    if S["wetter_ergebnis"]: p.append(f"Stärkste Beeinträchtigung in der Simulation: {S['wetter_ergebnis']['schlimmst']}.")
    if S["nebel_entscheidung"]: p.append("Reaktion auf schlechte Sicht: " + ("Schwellenwert senken." if S["nebel_entscheidung"] == "senken" else "Einsatzbereich begrenzen."))
    if S["angriff"]: p.append("Gezielter Angriff: " + (f"wirksam ab {S['angriff']['kleinste']} % Mustergröße." if S["angriff"]["kleinste"] else "in deinen Versuchen nicht wirksam."))
    with ab_zusammenfassung:
        ab_zusammenfassung.clear_output()
        if p: display(karte("<b>Deine Ergebnisse</b><ul>" + "".join(f"<li>{x}</li>" for x in p) + "</ul>", "info"))
for k in ab_knoepfe: k.on_click(ab_klick)
display(karte(f"<b>{FRAGE}</b>", "frage"), W.HBox(ab_knoepfe), ab_out, ab_zusammenfassung)

### Fachbegriffe

| Begriff | Englisch | Bedeutung |
|---|---|---|
| Objekterkennung | Object Detection | Ein Modell findet und benennt Objekte in einem Bild. |
| Erkennungskasten | Bounding Box | Rechteck um ein erkanntes Objekt. |
| Konfidenz | Confidence | Vom Modell berechneter Wert zwischen 0 und 1. Keine Wahrscheinlichkeit. |
| Schwellenwert | Threshold | Mindestkonfidenz, ab der eine Erkennung zählt. Wird von Menschen festgelegt. |
| Arbeitspunkt | Operating Point | Der für ein System gewählte Schwellenwert. |
| Referenz | Ground Truth | Die als richtig angenommene Lösung, hier deine Zählung. |
| Übersehene Person | False Negative | Eine vorhandene Person wird nicht erkannt. |
| Fehlalarm | False Positive | Eine Person wird gemeldet, wo keine ist. |
| Zielkonflikt | Trade-off | Weniger übersehene Personen bedeuten mehr Fehlalarme und umgekehrt. |
| Auslösende Bedingung | Triggering Condition | Umstand, unter dem ein System ohne Defekt Fehler macht (ISO 21448). |
| Einsatzbereich | Operational Design Domain (ODD) | Bedingungen, für die ein automatisiertes Fahrsystem ausgelegt ist. |
| Risikominimales Manöver | Minimal Risk Maneuver | Übergang in einen sicheren Zustand, etwa kontrolliertes Anhalten. |
| Verdeckung | Occlusion | Ein Objekt ist teilweise verdeckt. |
| Gezielter Angriff | Adversarial Attack | Berechnete Eingabe, die ein Modell gezielt täuscht. |
| Sensorfusion | Sensor Fusion | Kombination von Kamera, Radar und LiDAR, damit die Stärken eines Sensors die Schwächen eines anderen ausgleichen. |
| Safety | Functional Safety, SOTIF | Schutz vor Gefährdungen durch Fehler oder durch Leistungsgrenzen der Technik (ISO 26262, ISO 21448). |
| Security | Cybersecurity | Schutz vor absichtlichen Angriffen (ISO/SAE 21434). |
| Sicherheitsnachweis | Safety Case | Begründete Argumentation, warum ein System ausreichend sicher ist. |

### Wer arbeitet daran?

| Aufgabe aus diesem Notebook | Arbeitsgebiet |
|---|---|
| Arbeitspunkt festlegen und begründen | Entwicklung und Absicherung von Wahrnehmungssystemen (ISO 21448) |
| Verhalten bei schlechter Sicht prüfen | Test, Validierung und Simulation |
| Einsatzbereich festlegen | Systemauslegung und Sicherheitskonzept |
| Gezielte Angriffe untersuchen und abwehren | Automotive Security (ISO/SAE 21434) |
| Begründen, dass ein Fahrzeug sicher genug ist | Sicherheitsnachweis, Prüfung und Zulassung |

Diese Themen sind Teil von Studiengängen wie Informatik, Elektrotechnik oder Fahrzeugtechnik, im Bachelor und vertieft im Master.

### Weiterführend

- **Vertiefung:** [Sicher durch Zufall? Wie viel Testen reicht](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/00_vertiefung_wie_viel_testen_reicht.ipynb). Warum Zufallstests allein keine Sicherheit belegen.
- **Kursmaterial:** 34 englischsprachige Notebooks zu Wahrnehmung, Fehleranalyse, funktionaler Sicherheit, SOTIF, Security, KI-Sicherheit und Zulassung, [Übersicht im Repository](https://github.com/milinpatel07/campustage-2026-sicher-durch-zufall#kursmaterial).

---
Milin Patel · Hochschule Kempten (IFM) · MIT License. Beispielbild: Ultralytics (AGPL-3.0).